In [ ]:
# testin cell
# from openai import OpenAI
# c = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
# resp = c.chat.completions.create(
#     model="google/gemma-4-31B-it",
#     messages=[{"role": "user", "content": "List three primary colors. Respond as JSON: {\"colors\": [...]}"}],
#     max_tokens=4096,
#     extra_body={"chat_template_kwargs": {"enable_thinking": True}},
# )
# msg = resp.choices[0].message
# print("content:  ", repr(msg.content))
# print("reasoning:", repr(getattr(msg, "reasoning", None)))

In [1]:
import re
import json
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import os
import json
from datetime import datetime
from pathlib import Path

from openai import OpenAI

import numpy as np
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from label_studio_sdk import LabelStudio
import httpx
import pandas as pd

load_dotenv()

VLLM_URL = os.getenv("VLLM_URL", "http://localhost:8000/v1")
MODEL = "google/gemma-4-31B-it"

vllm_client = OpenAI(base_url=VLLM_URL, api_key="EMPTY")

def classify_messages(messages, max_tokens=64):
    """One chat-completion call against the vLLM OpenAI-compatible server."""
    resp = vllm_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content


# --- Thinking-mode helpers (Gemma 4) ---
# Server must be launched with `--reasoning-parser gemma4 --chat-template <path>/tool_chat_template_gemma4.jinja`.
# Thinking is OPT-IN per request via `chat_template_kwargs.enable_thinking=True`.
# vLLM exposes the parsed thought channel as `message.reasoning` (this build);
# `reasoning_content` is kept as a fallback for other servers/versions.
# Raw thought delimiters are asymmetric: `<|channel>thought ... <channel|>`.
_THOUGHT_RE = re.compile(r"<\|channel>\s*thought\s*\n?(.*?)<channel\|>", re.DOTALL)

def _split_thought(content):
    if "<|channel>" not in content:
        return content, ""
    m = _THOUGHT_RE.search(content)
    if not m:
        return content, ""
    reasoning = m.group(1).strip()
    cleaned = _THOUGHT_RE.sub("", content).strip()
    return cleaned, reasoning


def classify_messages_thinking(messages, max_tokens=4096):
    """Return (content, reasoning). Requires server with --reasoning-parser gemma4."""
    resp = vllm_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": True}},
    )
    msg = resp.choices[0].message
    content = msg.content or ""
    reasoning = (
        getattr(msg, "reasoning", None)
        or getattr(msg, "reasoning_content", None)
        or ""
    )
    if not reasoning:
        content, reasoning = _split_thought(content)
    return content, reasoning


def classify_batch(messages_list, max_tokens=64, max_workers=16):
    """Run many chat completions concurrently; preserves input order. Returns content strings."""
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(lambda m: classify_messages(m, max_tokens), messages_list))


def classify_batch_thinking(messages_list, max_tokens=4096, max_workers=16):
    """Concurrent thinking-mode batch. Returns list of (content, reasoning) tuples."""
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(lambda m: classify_messages_thinking(m, max_tokens), messages_list))


print(f"vLLM client → {VLLM_URL}, model={MODEL}")

vLLM client → http://localhost:8000/v1, model=google/gemma-4-31B-it


## Import annotations

- [Label Studio's API](https://labelstud.io/guide/api)
- [Task agreement](https://docs.humansignal.com/guide/stats.html)

In [2]:
ls_client = LabelStudio(
    base_url=os.getenv("LABEL_STUDIO_URL"),
    api_key=os.getenv("LABEL_STUDIO_API_KEY"),
    httpx_client=httpx.Client(verify=False),
)

PROJECT_ID = 110  # clean re-uploaded project, ground truth in annotations (union of both annotators)

all_tasks = list(ls_client.tasks.list(project=PROJECT_ID))
print(f"Found {len(all_tasks)} tasks in project {PROJECT_ID}")

# Per-task lookups used downstream by both location and topic sections
task_year = {t.id: t.data.get("Pub Year") for t in all_tasks}

# Tasks ready for classification (have title + abstract)
to_classify = [
    (t.id, t.data.get("Article Title"), t.data.get("text"))
    for t in all_tasks
    if t.data.get("Article Title") and t.data.get("text")
]
print(f"Classifiable (with title + text): {len(to_classify)}")

# Ground-truth annotations for eval — one row per task, all controls in one frame
rows = []
for t in all_tasks:
    if not t.annotations:
        continue
    controls = {}
    for r in t.annotations[0]["result"]:
        if r.get("type") == "choices":
            controls[r["from_name"]] = [v.lower() for v in r["value"]["choices"]]
    if not controls:
        continue
    rows.append({
        "task_id": t.id,
        "doi": t.data.get("DOI"),
        "locations_gt": controls.get("Location", []),
        "topics_gt": controls.get("topic", []),
        "methods_gt": controls.get("methods", []),
    })

gt = pd.DataFrame(rows)
print(f"Annotated tasks: {len(gt)}")
print(f"  with Location: {(gt['locations_gt'].map(len) > 0).sum()}")
print(f"  with topic:    {(gt['topics_gt'].map(len) > 0).sum()}")
print(f"  with methods:  {(gt['methods_gt'].map(len) > 0).sum()}")

Found 360 tasks in project 110
Classifiable (with title + text): 360
Annotated tasks: 88
  with Location: 88
  with topic:    87
  with methods:  86


In [3]:
CSV_PATH = "/gpfs1/home/j/s/jstonge1/rural-geog-classif/extract/input/Full Dataset Rur Geog WoS 1986-2025 4-28-2026.csv"

df = pd.read_csv(CSV_PATH, usecols=["Article Title", "Abstract", "DOI", "Pub Year", "Authors"], encoding='latin')
df = df.rename(columns={
    "Article Title": "title",
    "Abstract": "abstract",
    "DOI": "doi",
    "Pub Year": "year",
    "Authors": "authors",
})

before = len(df)
df = df.dropna(subset=["title", "abstract"])
df = df[df["abstract"].str.strip().astype(bool)]
print(f"Loaded {len(df)} papers with title+abstract (dropped {before - len(df)} missing)")

df.head(3)[["doi", "year", "authors", "title"]]

Loaded 360 papers with title+abstract (dropped 10 missing)


,doi,year,authors,title
0,10.1111/gere.12244,2018,"Abizaid, C; Coomes, OT; Takasaki, Y; Arroyo-Mo...",Rural Social Networks along Amazonian Rivers: ...
1,10.1080/00330124.2025.2582789,2025,"Adu-Poku, A; Appiah, IG; Kemausuor, F",Geographical Disparities in Energy Access: Cha...
2,10.1080/00045608.2014.985626,2015,"Aguayo, BC; Latta, A",Agro-Ecology and Food Sovereignty Movements in...


## 1. Classify locations

In [ ]:
SYSTEM_PROMPT = """You are an expert at classifying academic geography papers by the geographic location of their study area.

Given a paper's title and abstract, choose **exactly one** of the following labels:

- USA
- Other North America
- Europe
- Asia
- South America
- Africa
- Australia
- multiple regions
- unclear or conceptual

Guidelines:
- "USA" is reserved for studies focused on the United States. Use "Other North America" for Canada, Mexico, Central America, or the Caribbean.
- Russia and the post-Soviet states — including Siberia, the Caucasus, and the Caspian Sea region — are classified as **Europe** in this taxonomy, even though parts are geographically Asian.
- For papers covering several countries within one continent, use that continent's label.
- For transboundary studies within a continent (e.g., a USA–Mexico border study), pick the primary study area.
- Use "multiple regions" only for cross-continental or global comparative studies (e.g., a paper comparing the USA, Europe, and Asia).
- Use "unclear or conceptual" for theoretical, methodological, or review papers with no specific study area, or when the location cannot be determined from the title and abstract.
- Base your decision only on what is stated in the title and abstract — do not infer locations not mentioned.

Examples:

Input:
Fire is a fundamental tool within a broad spectrum of vegetation-management strategies, from swidden agriculture to plantation forestry. Through the seemingly pyromanic activity of incendiarism, fire assumes additional significance in the human-environment relationship. Case studies from England, Algeria, and the southern United States serve to illustrate the circumstance of fire as an indication of agrarian discontent and a weapon of peasant resistance. Other documented cases of incendiarism reveal that use of fire in the landscape has expanded from a constructive ecosystem-manipulation technique to a destructive form of protest undertaken by the oppressed or disempowered.
Output:
{"location": "multiple regions"}

Input:
This article examines the unintended outcomes of a neoliberal program designed to privatize Mexico's communal lands. Although postrevolutionary agrarian law excluded women from official landholding and leadership positions, steps toward land privatization inadvertently increased women's access to land, government resources, and political power. Using ethnographic and survey data collected in a Veracruz ejido, I demonstrate how Mexico's agrarian counterreforms triggered novel subjectivities and practices.
Output:
{"location": "Other North America"}

Input:
The Colorado River delta is a sedimentary alluvial formation that embodies the Lower Colorado River transboundary aquifer. The Mexicali Valley overlies the Mexican part of the aquifer, and the Imperial Valley the aquifer's portion north of the Mexico-U.S. border. This article presents a methodology applying remote sensing, geographic information analysis, and hydrologic analysis to calculate the annual water deficit in the Mexicali Valley. The work evaluates the valley's annual water deficit in reference to current agricultural and socioeconomic trends observed in the study region.
Output:
{"location": "Other North America"}

Input:
This paper examines migration in Russia during the period that preceded the breakup of the former Soviet Union and during the current transition period. The case study focuses on Yaroslavl Oblast and uses regional-level statistics to identify shifting patterns of inter-regional flows.
Output:
{"location": "Europe"}

Input:
This focus section aims to identify, conceptualize, and understand the emerging geographies of rural crime, in particular those of globalized rural crime, and evaluate their impact on different rural places. Contributions to this focus section span case studies from the United States, England, France, and Brazil to map the global countryside.
Output:
{"location": "multiple regions"}

Respond with a JSON object of the form:
{"location": "<label>"}

Use the exact label strings listed above. Do not include any other text.
"""

print("System prompt ready.")

In [ ]:
def classify_paper(title, abstract):
    """Classify a paper's study location from its title and abstract."""
    return classify_messages(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": f"Title: {title}\n\nAbstract: {abstract}"}],
        max_tokens=64,
    )

print("classify_paper() ready.")

In [ ]:
test = gt.merge(df, on="doi", how="inner")
test = test[test["locations_gt"].map(len) == 1].copy()
print(f"Test set: {len(test)} papers (single-label annotations only)")

messages_list = [
    [{"role": "system", "content": SYSTEM_PROMPT},
     {"role": "user", "content": f"Title: {t}\n\nAbstract: {a}"}]
    for t, a in zip(test["title"], test["abstract"])
]

texts = classify_batch(messages_list, max_tokens=64)

def parse_location(text):
    m = re.search(r"\{.*\}", text.strip(), re.DOTALL)
    return [json.loads(m.group())["location"]]

test["locations_pred"] = [parse_location(t) for t in texts]
test[["doi", "locations_gt", "locations_pred"]].head()

In [ ]:
LABELS = [
    "USA", "Other North America", "Europe", "Asia", "South America",
    "Africa", "Australia", "multiple regions", "unclear or conceptual",
]

M = np.zeros((len(LABELS), len(LABELS)), dtype=int)
for gt_labels, pred_labels in zip(test["locations_gt"], test["locations_pred"]):
    for g in gt_labels:
        for p in pred_labels:
            if g in LABELS and p in LABELS:
                M[LABELS.index(g), LABELS.index(p)] += 1

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(M, cmap="Blues")
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
ax.set_xlabel("Gemma prediction"); ax.set_ylabel("Annotator")
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        c = "white" if M[i, j] > M.max() / 2 else "black"
        ax.text(j, i, M[i, j], ha="center", va="center", color=c, fontsize=9)
fig.colorbar(im); fig.tight_layout()
plt.show()

print("\nRows = annotator (ground truth), Cols = Gemma prediction\n")
with pd.option_context("display.max_columns", None, "display.width", 200):
    print(pd.DataFrame(M, index=LABELS, columns=LABELS).to_string())

exact = sum(set(g) == set(p) for g, p in zip(test["locations_gt"], test["locations_pred"]))
print(f"\nExact-match (label set equality): {exact}/{len(test)} = {exact/len(test):.1%}")

In [ ]:
mismatches = test[
    test.apply(lambda r: set(r["locations_gt"]) != set(r["locations_pred"]), axis=1)
].copy()
print(f"Mismatches: {len(mismatches)}/{len(test)}\n")

for _, r in mismatches.head(10).iterrows():
    print(f"GT={r['locations_gt']}  PRED={r['locations_pred']}")
    print(f"  TITLE: {r['title']}")
    print(f"  ABSTR: {r['abstract'][:300]}...")
    print()

In [ ]:
print(f"Classifying {len(to_classify)} tasks for Location")

messages_list = [
    [{"role": "system", "content": SYSTEM_PROMPT},
     {"role": "user", "content": f"Title: {title}\n\nAbstract: {abstract}"}]
    for _, title, abstract in to_classify
]

texts = classify_batch(messages_list, max_tokens=64)

preds = []
for (task_id, _, _), text in zip(to_classify, texts):
    label = parse_location(text)[0]
    preds.append({
        "task": task_id,
        "result": [{
            "from_name": "Location",
            "to_name": "text",
            "type": "choices",
            "value": {"choices": [label]},
        }],
        "model_version": "gemma-4-31B-it-loc-v1",
    })

In [ ]:
results_df = pd.DataFrame([
    {"year": task_year[p["task"]],
     "label": p["result"][0]["value"]["choices"][0]}
    for p in preds
])
results_df = results_df.dropna(subset=["year"])
results_df["year"] = results_df["year"].astype(int)
results_df["bin"] = (results_df["year"] // 5) * 5

counts = results_df.groupby(["bin", "label"]).size().unstack(fill_value=0)
counts = counts.reindex(columns=[l for l in LABELS if l in counts.columns])
props = counts.div(counts.sum(axis=1), axis=0)

fig = plt.figure(figsize=(20, 7))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2], wspace=0.15)

# Left: stacked area
ax_area = fig.add_subplot(gs[0])
props.plot(kind="area", stacked=True, ax=ax_area, alpha=0.85)
ax_area.set_title("Stacked: composition per 5-year bin")
ax_area.set_xlabel("5-year bin")
ax_area.set_ylabel("Proportion")
ax_area.set_xticks(props.index)
ax_area.set_xticklabels([f"{b}-{b+4}" for b in props.index], rotation=45, ha="right")
ax_area.set_ylim(0, 1)
ax_area.legend(loc="upper left", bbox_to_anchor=(0, -0.18), ncol=3, fontsize=8)

# Right: 3x3 small multiples (shared y-axis for comparability)
gs_right = gs[1].subgridspec(3, 3, hspace=0.55, wspace=0.18)
y_max = props.values.max() * 1.1
for idx, label in enumerate(LABELS):
    r, c = divmod(idx, 3)
    ax = fig.add_subplot(gs_right[r, c])
    series = props[label] if label in props.columns else pd.Series(0.0, index=props.index)
    ax.plot(series.index, series.values, marker="o", linewidth=1.5)
    ax.fill_between(series.index, 0, series.values, alpha=0.25)
    ax.set_title(label, fontsize=9)
    ax.set_ylim(0, y_max)
    ax.set_xticks(props.index)
    if r == 2:
        ax.set_xticklabels([str(b)[2:] for b in props.index], fontsize=7)
    else:
        ax.set_xticklabels([])
    ax.tick_params(axis="y", labelsize=7)
    ax.grid(alpha=0.3)

fig.suptitle("Predicted study location of rural-geography papers (5-year bins)", fontsize=13, y=0.99)
plt.tight_layout()
plt.show()

print("\nPapers per 5-year bin:")
print(counts.sum(axis=1).to_string())

In [ ]:
print(f"Built {len(preds)} predictions, pushing to project {PROJECT_ID}...")
resp = ls_client.projects.import_predictions(id=PROJECT_ID, request=preds)
print(f"Response: {resp}")

## 2. Classify topics

In [ ]:
TOPIC_MAX_TOKENS = 4096  # token budget for the topic classifier (thinking eats tokens)

TOPIC_PROMPT = """You are an expert at classifying academic geography papers by topic.

Given a paper's title and abstract, identify which of the following topics the paper substantively covers. This is a **multi-label** task — select all that apply (usually 2–3 topics, rarely more than 4).

Topics:
- landscape
- gender
- migration
- agriculture
- climate change
- indigenous populations
- livelihood
- property
- mobility
- recreation/second homes
- inequality
- race
- political systems
- forests
- energy
- justice
- weather or hazards
- urbanization
- digital technology
- development
- resilience
- not rural?
- infrastructure
- policy
- water
- power
- class
- natural resources
- identity
- planning
- poverty
- animals or plants
- emotion

Guidelines:
- Use the exact label strings above. Capitalization matters ("Agriculture", "Climate Change", "Indigenous Populations" are capitalized; the rest are lowercase). The label "not rural?" includes the question mark.
- Pick topics that are CENTRAL to the paper, not peripheral mentions. A passing reference to "policymakers", "development", or "infrastructure" in framing or implications does NOT warrant those labels — only use them when the topic is a primary subject of analysis.
- Use "not rural?" when the paper is fundamentally about urban or non-rural settings and only tangentially (or not at all) about rural areas. In those cases, "not rural?" can be the only label.
- Some abstracts contain garbled or encoding-corrupted multilingual text (sequences of "?", mojibake, or repeated translations of the same content in Spanish/Chinese/etc.). Ignore the corrupted segments and base your classification on the readable English portion only.
- Base your decision only on what is stated in the title and abstract — do not infer topics not mentioned.

Examples:

Input:
This study examines geographical disparities in energy access across Ghana, analyzing the spatial patterns and socioeconomic drivers that influence energy distribution. Using a mixed-methods approach combining geospatial analysis with statistical evaluation of regional data, the research reveals significant disparities between northern and southern regions, as well as between urban and rural areas. Although Ghana maintains an impressive national electrification rate of 89 percent, access varies dramatically from 99 percent in the Greater Accra region to as low as 61 percent in the Savannah region. The study identifies key factors contributing to these disparities, including proximity to power generation facilities, population density, and economic activity levels. Analysis of clean cooking fuel access shows similar spatial patterns, with liquified petroleum gas adoption ranging from 69 percent in Greater Accra to minimal levels in northern regions. The study proposes pathways for a just energy transition, emphasizing the role of decentralized renewable energy systems, public-private partnerships, and targeted policy interventions. Drawing lessons from successful initiatives in Kenya and Rwanda, the study provides actionable recommendations for addressing spatial inequities in Ghana's energy landscape. The findings contribute to the broader discourse on achieving universal energy access while highlighting the importance of geographical considerations in energy policy and infrastructure development.
Output:
{"topics": ["inequality", "energy", "justice"]}

Input:
The African continent is portrayed in development texts as experiencing environmental crises of staggering proportions. Despite a lack of reliable data, the World Bank considers environmental degradation to be so widespread that the business of environmental planning and regulation is now seen as a global affair. It currently requires low-income countries receiving its financial assistance to develop National Environmental Action Plans (NEAPs) which, in assembly line fashion, are being produced according to a blueprint. Taking the West African case study of Cote d'Ivoire, this paper argues that the planning process, specifically the identification of environmental problems, is based on a poor understanding of the nature and direction of environmental change. We confront this data problem by contrasting the image of a deforested savanna landscape found in the Cote d'Ivoire NEAP with the more wooded landscape experienced by farmers and herders and confirmed by our analysis of aerial photographs. Our discussion of environmental change is informed by intensive data collection in two rural communities in the Korhogo region of northern Cote d'Ivoire. Research methods included focus-group discussions, household surveys, aerial photo analysis, GIS mapping, and vegetation transects.
Output:
{"topics": ["landscape", "livelihood", "natural resources"]}

Input:
Historical political ecology provides a powerful framework for understanding nature-society relations in the past. This approach is applied to municipal drinking water governance in early colonial Lima, Peru, with a focus on how power dynamics influenced sociospatial patterns of water access and control. Sixteenth- and seventeenth-century archival sources are analyzed for material aspects of Lima's drinking water pipeline network and for the management strategies employed by the municipal government. Access to water is demonstrated to have shaped, reinforced, and reflected colonial social divisions and to have been linked to the spatial development of the city, including urban-rural relations. ???????,????????????,?????????????????????????????????????????? La ecologia politica historica nos proporciona una potente armazon para comprender mejor las relaciones naturaleza-sociedad del pasado. Se aplico este enfoque al estudio de la administracion municipal del agua potable en Lima, Peru, a principios de la colonia.
Output:
{"topics": ["inequality", "development", "infrastructure", "water"]}

Input:
For a discipline so oriented around the study of wakeful geographies, the lack of direct conceptual engagement with the notion of wakefulness — a cognitive state in which the mind is conscious of, and responsive to, the external world — is all-the-more remarkable. In this article we advance geography by probing and revealing the links that exist between wakefulness and indebted life under capitalism. We show how villagers in rural Cambodia experience wakefulness in their day and nighttime lives through a punitive alertness to, and excessive rumination on, the pressures and challenges they face to repay microfinance debts on time. Capitalist debt demands a keen alertness and submission to the clock time of repayment obligations, and with this fosters a hyperalertness to debt that works to the consequential exclusion of sleep. The article evidences the enforced isorhythmic alignment between the demands of creditors and the bodies and minds of borrowers. This claim is evidenced further through the arrhythmic impacts of the COVID-19 pandemic and the intensifying climate crisis, which are further undermining borrowers' hopes for debt-free lives.
Output:
{"topics": ["climate change", "power", "class"]}

Input:
Sociospatial segregation has long been a critical subject in urban studies. In recent decades, increasing attention has been given to the segregation experienced in activity spaces beyond the well-examined residential locations. A large body of studies revealed that disadvantaged groups (e.g., low-income migrants) can access more opportunities for cross-group interaction by engaging in activities outside of their residential areas. This study extends this body of research by examining whether individuals have to travel a certain distance before their activity-space exposure noticeably increases, with a focus on low-income migrants in Shenzhen, China. Through analyzing the nonlinear relationship between travel distance and activity-space exposure using cellphone data and gradient boosting decision tree (GBDT) models, the study confirms the possibility of increased interactions in activity spaces and reveals that low-income migrants must travel a threshold distance before their activity-space exposure significantly increases. Central-city areas offer shorter distance opportunities for diverse social interactions due to higher population density and proximity to urban amenities, whereas suburban areas necessitate longer travel.
Output:
{"topics": ["not rural?"]}

Input:
This article explores the regional disparities in water supply coverage in Pakistan in the context of intraregional development patterns. We applied the Gini index, Esteban and Ray index, and Moran's I to measure regional development patterns in terms of inequality, polarization, and spatial concentration and build an integrated framework combining them with other widely accepted factors to understand regional differences in water coverage quantitatively. Our results show that improved water (tap and pump water) is more widely supplied in Pakistan with lower inequality but higher spatial concentration than tap water supply coverage, although the patterns differ by urban-rural divisions and provinces. The regression models show various mechanisms of improved water and tap water coverage. Both are related to local economic conditions but tap water coverage is more related to local environmental conditions, and improved water is more related to local demographic conditions. Urban planners and policymakers should address polycentric regional development to fulfill various development targets and improve regional equity of drinking water coverage in terms of accessibility, availability, and quality.
Output:
{"topics": ["inequality", "water", "natural resources", "planning"]}

Respond with a JSON object of the form:
{"topics": ["<label>", ...]}

Use the exact label strings listed above. Do not include any other text.
"""

print(f"Topic prompt ready (max_tokens={TOPIC_MAX_TOKENS}).")

In [ ]:
gt_topic = gt[gt["topics_gt"].map(len) > 0].copy()
print(f"Tasks with a topic annotation: {len(gt_topic)}")
print(f"Mean # topics per paper: {gt_topic['topics_gt'].map(len).mean():.2f}")
print("\nMost common topic labels:")
print(gt_topic["topics_gt"].explode().value_counts().head(15))

In [ ]:
test_topic = gt_topic.merge(df, on="doi", how="inner")
print(f"Test set: {len(test_topic)} papers")

messages_list = [
    [{"role": "system", "content": TOPIC_PROMPT},
     {"role": "user", "content": f"Title: {t}\n\nAbstract: {a}"}]
    for t, a in zip(test_topic["title"], test_topic["abstract"])
]

# Thinking mode: capture both the JSON answer and the model's thought channel
results = classify_batch_thinking(messages_list, max_tokens=TOPIC_MAX_TOKENS)

# --- Inspect the first raw response so we know what we're parsing ---
print("\n=== First response — content (first 400 chars) ===")
print(repr(results[0][0][:400]))
print("\n=== First response — reasoning (first 400 chars) ===")
print(repr(results[0][1][:400]))

def parse_topics(text):
    m = re.search(r"\{.*?\}", text.strip(), re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group())["topics"]
    except (json.JSONDecodeError, KeyError):
        return None

test_topic["topics_pred"] = [parse_topics(c) for c, _ in results]
test_topic["reasoning"]   = [r for _, r in results]

n_unparsed = test_topic["topics_pred"].isna().sum()
n_with_reasoning = (test_topic["reasoning"].str.len() > 0).sum()
print(f"\nUnparsed responses: {n_unparsed} / {len(test_topic)}")
print(f"Got reasoning for:  {n_with_reasoning} / {len(test_topic)}")

# Show one unparsed example if any
if n_unparsed:
    bad_idx = test_topic[test_topic["topics_pred"].isna()].index[0]
    print(f"\n=== First unparsed response (row {bad_idx}) ===")
    print(repr(results[list(test_topic.index).index(bad_idx)][0][:600]))

test_topic[["doi", "topics_gt", "topics_pred"]].head()

In [ ]:
# Drop unparsed rows before computing metrics so None preds don't crash set ops
n_total = len(test_topic)
test_topic_valid = test_topic.dropna(subset=["topics_pred"]).copy()
n_dropped = n_total - len(test_topic_valid)
if n_dropped:
    print(f"Dropped {n_dropped} rows with unparsed predictions before scoring\n")

def jaccard(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 1.0
    return len(sa & sb) / len(sa | sb)

jaccards = [jaccard(g, p) for g, p in zip(test_topic_valid["topics_gt"], test_topic_valid["topics_pred"])]
exact = sum(set(g) == set(p) for g, p in zip(test_topic_valid["topics_gt"], test_topic_valid["topics_pred"]))

print(f"Mean Jaccard:  {sum(jaccards)/len(jaccards):.3f}")
print(f"Exact-match:   {exact}/{len(test_topic_valid)} = {exact/len(test_topic_valid):.1%}")
print(f"Mean |gt|:     {test_topic_valid['topics_gt'].map(len).mean():.2f}")
print(f"Mean |pred|:   {test_topic_valid['topics_pred'].map(len).mean():.2f}")

# Per-label support / hit / precision / recall / F1
support, predicted, hit = defaultdict(int), defaultdict(int), defaultdict(int)
for g, p in zip(test_topic_valid["topics_gt"], test_topic_valid["topics_pred"]):
    sg, sp = set(g), set(p)
    for label in sg:
        support[label] += 1
        if label in sp:
            hit[label] += 1
    for label in sp:
        predicted[label] += 1

stats = pd.DataFrame({
    "support": pd.Series(support),
    "predicted": pd.Series(predicted),
    "hit": pd.Series(hit),
}).fillna(0).astype(int)
stats["recall"] = stats["hit"] / stats["support"].replace(0, 1)
stats["precision"] = stats["hit"] / stats["predicted"].replace(0, 1)
stats["f1"] = 2 * stats["precision"] * stats["recall"] / (stats["precision"] + stats["recall"]).replace(0, 1)
stats[["recall", "precision", "f1"]] = stats[["recall", "precision", "f1"]].round(2)

labels_with_support = stats[stats["support"] > 0]
macro_f1 = labels_with_support["f1"].mean()
print(f"Macro-F1:      {macro_f1:.3f}  (mean per-label F1 across {len(labels_with_support)} labels with gt support)")

print("\nPer-label stats (sorted by gt support):")
print(stats.sort_values("support", ascending=False).to_string())

In [ ]:
# Per-label confusion: hits (TP), false positives (over-tagged), false negatives (missed)
viz = stats[stats["support"] > 0].sort_values("support", ascending=True)

hits_v = viz["hit"].values
fps_v  = (viz["predicted"] - viz["hit"]).values  # predicted but not in gt
fns_v  = (viz["support"]   - viz["hit"]).values  # in gt but not predicted

fig, ax = plt.subplots(figsize=(10, max(6, 0.32 * len(viz))))
ax.barh(viz.index, hits_v, color="seagreen", label="hit (correct)")
ax.barh(viz.index, fps_v,  left=hits_v,           color="indianred",  label="false positive (over-tagged)")
ax.barh(viz.index, fns_v,  left=hits_v + fps_v,   color="goldenrod",  label="false negative (missed)")

# Annotate each bar with f1
for label in viz.index:
    row = viz.loc[label]
    total = row["hit"] + (row["predicted"] - row["hit"]) + (row["support"] - row["hit"])
    ax.text(total + 0.3, label, f"f1={row['f1']:.2f}", va="center", fontsize=8)

ax.set_xlabel("Count of label occurrences")
ax.set_title("Per-label classification breakdown (sorted by gt support, ascending)")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from textwrap import wrap

In [ ]:

# Drill-down: worst mismatches by per-paper Jaccard, including the model's reasoning
test_topic = test_topic.copy()
test_topic["jaccard"] = [
    jaccard(g, p) if p is not None else float("nan")
    for g, p in zip(test_topic["topics_gt"], test_topic["topics_pred"])
]

worst = test_topic.dropna(subset=["jaccard"]).sort_values("jaccard").tail(1)
print(f"Worst {len(worst)} mismatches by per-paper Jaccard:\n")
for _, r in worst.iterrows():
    gt_s, pr_s = set(r["topics_gt"]), set(r["topics_pred"])
    print(f"--- Jaccard={r['jaccard']:.2f} ---")
    print(f"  TITLE:  {r['title']}")
    print(f"  GT:     {sorted(gt_s)}")
    print(f"  PRED:   {sorted(pr_s)}")
    if gt_s - pr_s:
        print(f"  MISSED: {sorted(gt_s - pr_s)}")
    if pr_s - gt_s:
        print(f"  EXTRA:  {sorted(pr_s - gt_s)}")
    if r.get("reasoning"):
        print(f"  THOUGHT: {r['reasoning']}")
    print(f"  ABSTR:  {'\n'.join(wrap(r['abstract'],150))}")
    print()

In [ ]:
RUNS_DIR = Path("/gpfs1/home/j/s/jstonge1/rural-geog-classif/transform/src/runs")

def save_run(name, prompt, config, summary, per_label_df, predictions_df=None):
    """Persist a classification run (one directory per attempt)."""
    run_dir = RUNS_DIR / name
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "prompt.txt").write_text(prompt)
    (run_dir / "config.json").write_text(json.dumps(config, indent=2))
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    per_label_df.to_csv(run_dir / "per_label.csv")
    if predictions_df is not None:
        predictions_df.to_parquet(run_dir / "predictions.parquet")
    print(f"Saved run → {run_dir}")
    return run_dir

def compare_runs(runs_dir=RUNS_DIR):
    """Aggregate summary.json + config.json from every subdir into one leaderboard frame."""
    rows = []
    for d in sorted(Path(runs_dir).iterdir()):
        if not d.is_dir():
            continue
        s = d / "summary.json"
        if not s.exists():
            continue
        c = d / "config.json"
        config = json.loads(c.read_text()) if c.exists() else {}
        rows.append({"run": d.name, **config, **json.loads(s.read_text())})
    return pd.DataFrame(rows)


# --- Save the current topic eval ---
# Bump the suffix (v1, v2, ...) when changing the prompt/config so runs don't overwrite.
RUN_NAME = f"{datetime.now().strftime('%Y-%m-%d')}_topic_v2_thinking"

save_run(
    name=RUN_NAME,
    prompt=TOPIC_PROMPT,
    config={
        "model": MODEL,
        "control": "topic",
        "thinking": True,
        "max_tokens": TOPIC_MAX_TOKENS,
        "temperature": 0.0,
    },
    summary={
        "n_test":         len(test_topic),
        "n_unparsed":     int(test_topic["topics_pred"].isna().sum()),
        "n_scored":       len(test_topic_valid),
        "mean_jaccard":   round(sum(jaccards) / len(jaccards), 3),
        "exact_match":    round(exact / len(test_topic_valid), 4),
        "macro_f1":       round(macro_f1, 3),
        "mean_gt_size":   round(test_topic_valid["topics_gt"].map(len).mean(), 2),
        "mean_pred_size": round(test_topic_valid["topics_pred"].map(len).mean(), 2),
    },
    per_label_df=stats,
    predictions_df=test_topic_valid[[c for c in ["doi", "topics_gt", "topics_pred", "jaccard"] if c in test_topic_valid.columns]],
)

# Show the leaderboard so far
compare_runs()

In [ ]:
TOPIC_LABELS = [
    "landscape", "gender", "migration", "Agriculture", "Climate Change",
    "Indigenous Populations", "Livelihood", "Property", "mobility",
    "recreation/second homes", "inequality", "race", "political systems",
    "forests", "energy", "justice", "weather or hazards", "urbanization",
    "digital technology", "development", "resilience", "not rural?",
    "infrastructure", "policy", "water", "power", "class", "natural resources",
    "identity", "planning", "poverty", "animals or plants", "emotion",
]

# Predict topics for ALL tasks and push to project
messages_list = [
    [{"role": "system", "content": TOPIC_PROMPT},
     {"role": "user", "content": f"Title: {title}\n\nAbstract: {abstract}"}]
    for _, title, abstract in to_classify
]

texts = classify_batch(messages_list, max_tokens=256)

allowed = set(TOPIC_LABELS)
topic_preds = []
n_dropped = 0
for (task_id, _, _), text in zip(to_classify, texts):
    raw = parse_topics(text)
    cleaned = [t for t in raw if t in allowed]  # drop hallucinated labels
    n_dropped += len(raw) - len(cleaned)
    if not cleaned:
        continue  # skip predictions that resulted in 0 valid labels
    topic_preds.append({
        "task": task_id,
        "result": [{
            "from_name": "topic",
            "to_name": "text",
            "type": "choices",
            "value": {"choices": cleaned},
        }],
        "model_version": "gemma-4-31B-it-topic-v1",
    })

print(f"Built {len(topic_preds)} topic predictions ({n_dropped} non-canonical labels dropped), pushing to project {PROJECT_ID}...")

In [ ]:
resp = ls_client.projects.import_predictions(id=PROJECT_ID, request=topic_preds)
print(f"Response: {resp}")

In [ ]:
# Time-series of top topics across 5-year bins (multi-label, so doesn't sum to 1)
def to_bin(year):
    if pd.isna(year):
        return None
    return (int(year) // 5) * 5

bin_per_task = {tid: to_bin(y) for tid, y in task_year.items()}

rows_t = []
for tp in topic_preds:
    b = bin_per_task.get(tp["task"])
    if b is None:
        continue
    for label in tp["result"][0]["value"]["choices"]:
        rows_t.append({"bin": b, "label": label})

results_topic_df = pd.DataFrame(rows_t)
total_per_bin = pd.Series([bin_per_task[tp["task"]] for tp in topic_preds]).dropna().astype(int).value_counts().sort_index()

counts_topic = results_topic_df.groupby(["bin", "label"]).size().unstack(fill_value=0)
props_topic = counts_topic.div(total_per_bin, axis=0).fillna(0)

# Show the 9 most prevalent topics overall
top_topics = counts_topic.sum().nlargest(9).index.tolist()

fig, axes = plt.subplots(3, 3, figsize=(15, 9), sharey=True)
y_max = props_topic[top_topics].values.max() * 1.1
for ax, topic in zip(axes.flat, top_topics):
    series = props_topic[topic]
    ax.plot(series.index, series.values, marker="o", linewidth=1.5)
    ax.fill_between(series.index, 0, series.values, alpha=0.25)
    ax.set_title(topic, fontsize=10)
    ax.set_ylim(0, y_max)
    ax.grid(alpha=0.3)
fig.suptitle("Top 9 topics: share of papers per 5-year bin", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Classify methodology

In [4]:
METHODS_PROMPT = """You are an expert at classifying academic geography papers by methodology.

Given a paper's title and abstract, choose **exactly one** of the following labels for the paper's primary methodological approach:

- qual: qualitative methods (interviews, ethnography, focus groups, archival analysis, discourse analysis, single in-depth case study)
- quant: quantitative methods (statistical analysis, regression, large-n surveys, modeling, formal hypothesis testing)
- both: mixed methods — uses both qualitative and quantitative approaches substantively
- descriptive: descriptive or narrative analysis without formal statistical testing or in-depth qualitative inquiry (e.g., literature review, narrative synthesis, conceptual essay with illustrative cases)
- spatial/mapping: methodology centered on GIS, remote sensing, spatial analysis, cartography, or geospatial modeling
- unclear: methodology cannot be determined from the title and abstract

Guidelines:
- Use "both" only when the paper explicitly combines qualitative and quantitative methods — e.g., "we combine in-depth interviews with statistical analysis of survey data".
- Use "spatial/mapping" when the analytical method is **fundamentally spatial**: GIS, remote sensing, cartography, geospatial modeling, OR a spatial statistical method where geography is built into the model structure — geographically weighted regression (GWR), spatial autocorrelation / Moran's I, kriging, spatial lag/error models, point-pattern analysis, or remote-sensing image classification.
- DO NOT use "spatial/mapping" when the paper just applies standard statistics (OLS regression, t-tests, ANOVA, descriptive statistics) to data that happens to be geographic. Geographic framing alone ("spatial perspective", "platial perspective", "geographic patterns") is also not enough — the analysis itself must be spatial.
- "descriptive" is for papers that actually do description/synthesis with a clear deliverable — e.g., a literature review that surveys a defined corpus, a conceptual essay built around named theoretical frameworks, an agenda-setting piece that compares or summarizes specific cases.
- Use "unclear" when the abstract speaks at a high level about "reviewing", "deepening", "expanding", or "inviting" engagement WITHOUT naming a data source, a corpus, an analytical method, or specific case material. Position papers that gesture at "selected sites" or "reflections" but never say how they were studied are "unclear", not "descriptive".

Examples:

Input:
The aim of this article is to investigate the nature of information sharing in social media about missing persons by using social media data (mostly Twitter) and conventional media coverage (media archives), adopting a platial perspective to this geographical information. By focusing on the cases of three people gone missing in rural Sweden, the article analyzes message timelines and information cascades.
Output:
{"method": "quant"}

Input:
This study advances understanding of nature-society interactions by examining the spatiotemporal coupling of tsunami hazards and human responses. Using video footage of the 2011 Tohoku tsunami recorded in a rural coastal plain in Japan enabled analysis of inundation patterns and evacuation responses under different lead times. Multiple regression and geographically weighted regression analyses revealed that inundation patterns were predominantly controlled by coastal proximity and surface roughness, while road and waterway configurations locally modified flow velocities. Evacuation analysis identified distinct response patterns associated with different temporal zones of tsunami inundation.
Output:
{"method": "spatial/mapping"}

Input:
There has been a surge in references to urban geopolitics over the last twenty-plus years. Reviewing claims that warfare has urbanized, however, yields questions about the delineation and frontiers of the urban. The multiple meanings and definitions of geopolitics and urban beckon a broadening range of sites and an analytical deepening of urban geopolitics. To these ends, reviewing and seeking to develop the field in conceptual terms, the article revisits rural-urban interactions and reflections from selected African and Asian sites. The overall aim of the article is not to delimit urban geopolitics, but to deepen and expand agendas, broadening the range of cases that inform these. This also invites deeper geopolitical engagement with literature on extended and planetary urbanization.
Output:
{"method": "unclear"}

Respond with a JSON object of the form:
{"method": "<label>"}

Use the exact label strings listed above. Do not include any other text.
"""

print("Methods prompt ready.")

Methods prompt ready.


In [5]:
METHODS_LABELS = ["qual", "quant", "both", "descriptive", "spatial/mapping", "unclear"]

# Standalone thinking-mode methods eval — does not depend on the no-thinking cells.
test_methods_t = gt[gt["methods_gt"].map(len) > 0].merge(df, on="doi", how="inner")
test_methods_t = test_methods_t[test_methods_t["methods_gt"].map(len) == 1].copy()
print(f"Test set: {len(test_methods_t)} papers")

messages_list = [
    [{"role": "system", "content": METHODS_PROMPT},
     {"role": "user", "content": f"Title: {t}\n\nAbstract: {a}"}]
    for t, a in zip(test_methods_t["title"], test_methods_t["abstract"])
]

Test set: 84 papers


In [7]:
# Full text + thinking eval — uses parse/output/docling/{doi.replace('/','_')}.md when available,
# falls back to abstract if the docling file is missing.
DOCLING_DIR = Path("/gpfs1/home/j/s/jstonge1/rural-geog-classif/parse/output/docling")
FULLTEXT_MAX_CHARS = 12000  # ~3k tokens of input, leaves room for thinking

def load_docling(doi):
    if not isinstance(doi, str):
        return None
    path = DOCLING_DIR / (doi.replace("/", "_") + ".md")
    return path.read_text() if path.exists() else None

In [9]:
def parse_method(text):
    m = re.search(r"\{.*\}", text.strip(), re.DOTALL)
    if not m:
        return None
    try:
        return [json.loads(m.group())["method"]]
    except (json.JSONDecodeError, KeyError):
        return None

### Abstract-only

In [12]:
results_t = classify_batch_thinking(messages_list, max_tokens=2048)


test_methods_t["methods_pred"]      = [parse_method(c) for c, _ in results_t]
test_methods_t["methods_reasoning"] = [r for _, r in results_t]

n_unparsed = test_methods_t["methods_pred"].isna().sum()
print(f"Unparsed: {n_unparsed} / {len(test_methods_t)}")
print(f"Got reasoning for: {(test_methods_t['methods_reasoning'].str.len() > 0).sum()} / {len(test_methods_t)}")

valid_t = test_methods_t.dropna(subset=["methods_pred"]).copy()

Unparsed: 1 / 84
Got reasoning for: 84 / 84


In [ ]:
M_t = np.zeros((len(METHODS_LABELS), len(METHODS_LABELS)), dtype=int)
for gt_labels, pred_labels in zip(valid_t["methods_gt"], valid_t["methods_pred"]):
    for g in gt_labels:
        for p in pred_labels:
            if g in METHODS_LABELS and p in METHODS_LABELS:
                M_t[METHODS_LABELS.index(g), METHODS_LABELS.index(p)] += 1

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(M_t, cmap="Blues")
ax.set_xticks(range(len(METHODS_LABELS))); ax.set_xticklabels(METHODS_LABELS, rotation=45, ha="right")
ax.set_yticks(range(len(METHODS_LABELS))); ax.set_yticklabels(METHODS_LABELS)
ax.set_xlabel("Gemma prediction (thinking)"); ax.set_ylabel("Annotator")
ax.set_title("Methods — thinking")
for i in range(len(METHODS_LABELS)):
    for j in range(len(METHODS_LABELS)):
        c = "white" if M_t[i, j] > M_t.max() / 2 else "black"
        ax.text(j, i, M_t[i, j], ha="center", va="center", color=c, fontsize=10)
fig.colorbar(im); fig.tight_layout()
plt.show()

print("\nRows = annotator, Cols = Gemma (thinking)\n")
print(pd.DataFrame(M_t, index=METHODS_LABELS, columns=METHODS_LABELS).to_string())

exact_t = sum(set(g) == set(p) for g, p in zip(valid_t["methods_gt"], valid_t["methods_pred"]))
print(f"\nExact-match (thinking): {exact_t}/{len(valid_t)} = {exact_t/len(valid_t):.1%}")

In [ ]:
# Mismatches in the thinking run — group by (gt, pred) pair to see the worst confusions
mismatches_m = valid_t[
    valid_t.apply(lambda r: set(r["methods_gt"]) != set(r["methods_pred"]), axis=1)
].copy()
mismatches_m["pair"] = mismatches_m.apply(
    lambda r: f"{r['methods_gt'][0]} → {r['methods_pred'][0]}", axis=1
)
print(f"Mismatches: {len(mismatches_m)}/{len(valid_t)}\n")
print("Confusion pairs (gt → pred), most common first:")
print(mismatches_m["pair"].value_counts().to_string())
print()

# Print a few examples from the worst-offender pairs
for pair in mismatches_m["pair"].value_counts().head(4).index:
    print(f"\n{'='*80}\n{pair}\n{'='*80}")
    for _, r in mismatches_m[mismatches_m["pair"] == pair].head(2).iterrows():
        print(f"\nTITLE: {r['title']}")
        print(f"GT:    {r['methods_gt']}")
        print(f"PRED:  {r['methods_pred']}")
        print(f"ABSTR: {r['abstract'][:300]}...")
        if r.get("methods_reasoning"):
            print(f"THOUGHT: {r['methods_reasoning']}")

In [ ]:
# Just the DOIs of the mismatches, grouped by confusion pair — useful for looking up
# full-text files in parse/output/docling/ (docling uses underscores instead of slashes).
def doi_to_docling_path(doi):
    return doi.replace("/", "_") + ".md" if isinstance(doi, str) else None

print(f"{len(mismatches_m)} mismatches — DOIs by confusion pair:\n")
for pair, group in mismatches_m.groupby("pair"):
    print(f"  {pair}  ({len(group)})")
    for _, r in group.iterrows():
        print(f"    {r['doi']}    → docling: {doi_to_docling_path(r['doi'])}")
    print()

### Using full texts over abstracts (first X tokens)

In [ ]:

test_methods_ft = test_methods_t.copy()
test_methods_ft["fulltext"] = test_methods_ft["doi"].map(load_docling)
n_with_ft = test_methods_ft["fulltext"].notna().sum()
print(f"Full text available: {n_with_ft} / {len(test_methods_ft)} papers")

def build_user_msg(row):
    if row["fulltext"]:
        return f"Title: {row['title']}\n\nFull text (possibly truncated):\n{row['fulltext'][:FULLTEXT_MAX_CHARS]}"
    return f"Title: {row['title']}\n\nAbstract: {row['abstract']}"

messages_list = [
    [{"role": "system", "content": METHODS_PROMPT},
     {"role": "user", "content": build_user_msg(r)}]
    for _, r in test_methods_ft.iterrows()
]

results_ft = classify_batch_thinking(messages_list, max_tokens=4096)

test_methods_ft["methods_pred"]      = [parse_method(c) for c, _ in results_ft]
test_methods_ft["methods_reasoning"] = [r for _, r in results_ft]

n_unparsed = test_methods_ft["methods_pred"].isna().sum()
print(f"Unparsed: {n_unparsed} / {len(test_methods_ft)}")

valid_ft = test_methods_ft.dropna(subset=["methods_pred"]).copy()

M_ft = np.zeros((len(METHODS_LABELS), len(METHODS_LABELS)), dtype=int)
for gt_labels, pred_labels in zip(valid_ft["methods_gt"], valid_ft["methods_pred"]):
    for g in gt_labels:
        for p in pred_labels:
            if g in METHODS_LABELS and p in METHODS_LABELS:
                M_ft[METHODS_LABELS.index(g), METHODS_LABELS.index(p)] += 1

print("\nRows = annotator, Cols = Gemma (full text + thinking)\n")
print(pd.DataFrame(M_ft, index=METHODS_LABELS, columns=METHODS_LABELS).to_string())

exact_ft = sum(set(g) == set(p) for g, p in zip(valid_ft["methods_gt"], valid_ft["methods_pred"]))
print(f"\nExact-match (full text + thinking): {exact_ft}/{len(valid_ft)} = {exact_ft/len(valid_ft):.1%}")

# How many predictions changed vs the abstract-only thinking run?
joined = valid_t.merge(valid_ft, on="doi", suffixes=("_abstract", "_fulltext"))
flipped = joined[joined["methods_pred_abstract"].astype(str) != joined["methods_pred_fulltext"].astype(str)]
print(f"\nPredictions changed between abstract and full text: {len(flipped)} / {len(joined)}")

Full text available: 65 / 84 papers
Unparsed: 0 / 84

Rows = annotator, Cols = Gemma (full text + thinking)

                 qual  quant  both  descriptive  spatial/mapping  unclear
qual               21      0     1            1                0        1
quant               1     16     1            2                5        0
both                0      1     5            1                1        0
descriptive         4      1     0            5                0        1
spatial/mapping     0      1     1            0                3        1
unclear             3      2     0            2                0        3

Exact-match (full text + thinking): 53/84 = 63.1%

Predictions changed between abstract and full text: 9 / 84


### multi-phase approach

- Phase A (no LLM): parse_sections() pulls every # … ###### header out of the docling markdown and builds a {header_label → body_text} dict per paper.
- Phase B (1 LLM call/paper, no thinking, max_tokens=256): PICK_SYS prompt asks gemma to pick up to 4 verbatim headers that describe methodology/data/study area/research design. Outputs {"sections": [...]}. Only fires for papers that actually have headers.
- Phase C (1 LLM call/paper, thinking on, max_tokens=4096): concatenates the picked sections' bodies (capped at 4000 chars each) with title + abstract and runs the regular METHODS_PROMPT classifier.
- Diagnostics: prints how many papers have full text vs headers, distribution of how many headers got picked, unparsed count, confusion matrix, exact-match, and flip count vs abstract-only.

In [13]:
# Two-phase methods eval: (A) list section headers from docling, (B) ask gemma which sections
# describe methodology, (C) classify using abstract + those sections.

HEADER_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*$", re.MULTILINE)

def parse_sections(text):
    """Return dict: header_label (lowercased, no #s) -> body text."""
    matches = list(HEADER_RE.finditer(text))
    sections = {}
    for i, m in enumerate(matches):
        body_start = m.end()
        body_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        label = m.group(2).strip()
        sections[label] = text[body_start:body_end].strip()
    return sections

PICK_SYS = """You are helping classify the methodology of an academic geography paper.
Given the paper's title, abstract, and the list of section headers in its full text,
identify which sections (by header name, verbatim) describe the methodology, data, study area,
research design, or analytical approach.

Respond with a JSON object: {"sections": ["<header>", "<header>", ...]}
- Pick at most 4 headers, in order of priority.
- Use the exact header strings from the list.
- If no header in the list looks methodology-related, return {"sections": []}.
- Do not include any other text.
"""

SECTION_MAX_CHARS = 4000  # per-section truncation cap when feeding back to phase C

def build_pick_prompt(title, abstract, headers):
    return [
        {"role": "system", "content": PICK_SYS},
        {"role": "user", "content": (
            f"Title: {title}\n\nAbstract: {abstract}\n\n"
            "Section headers (verbatim):\n- " + "\n- ".join(headers)
        )},
    ]

def parse_picked(text):
    m = re.search(r"\{.*\}", text.strip(), re.DOTALL)
    if not m:
        return []
    try:
        return json.loads(m.group()).get("sections", []) or []
    except json.JSONDecodeError:
        return []


# --- Phase A: load docling and parse headers for each test paper ---
test_methods_2 = test_methods_t.copy()
test_methods_2["fulltext"] = test_methods_2["doi"].map(load_docling)
test_methods_2["sections"] = test_methods_2["fulltext"].map(lambda t: parse_sections(t) if t else {})

n_with_ft   = test_methods_2["fulltext"].notna().sum()
n_with_hdr  = (test_methods_2["sections"].map(len) > 0).sum()
print(f"Full text:    {n_with_ft} / {len(test_methods_2)}")
print(f"Has headers:  {n_with_hdr} / {len(test_methods_2)}")


# --- Phase B: ask gemma to pick methodology-related headers (1 LLM call per paper) ---
pick_msgs = [
    build_pick_prompt(r["title"], r["abstract"], list(r["sections"].keys()))
    for _, r in test_methods_2.iterrows()
]
# Only fire phase B for papers that actually have headers
fire_b = test_methods_2["sections"].map(len) > 0
picked_texts = [None] * len(test_methods_2)
fire_idx = [i for i, fb in enumerate(fire_b) if fb]
fire_msgs = [pick_msgs[i] for i in fire_idx]
fire_texts = classify_batch(fire_msgs, max_tokens=256)
for i, t in zip(fire_idx, fire_texts):
    picked_texts[i] = t

test_methods_2["picked"] = [parse_picked(t) if t else [] for t in picked_texts]
print(f"\nPhase B done. Distribution of #headers picked:")
print(test_methods_2["picked"].map(len).value_counts().sort_index().to_string())


# --- Phase C: build user message with extracted sections, classify ---
def build_classify_msg(row):
    if not row["picked"]:
        # Fall back to abstract if no headers picked or no full text
        return f"Title: {row['title']}\n\nAbstract: {row['abstract']}"
    parts = [f"Title: {row['title']}", f"Abstract: {row['abstract']}"]
    sections = row["sections"]
    for h in row["picked"]:
        body = sections.get(h)
        if body is None:
            # Try case-insensitive match
            for k, v in sections.items():
                if k.lower() == h.lower():
                    body = v
                    break
        if body:
            parts.append(f"\n## {h}\n{body[:SECTION_MAX_CHARS]}")
    return "\n\n".join(parts)

messages_list = [
    [{"role": "system", "content": METHODS_PROMPT},
     {"role": "user", "content": build_classify_msg(r)}]
    for _, r in test_methods_2.iterrows()
]

results_2 = classify_batch_thinking(messages_list, max_tokens=4096)
test_methods_2["methods_pred"]      = [parse_method(c) for c, _ in results_2]
test_methods_2["methods_reasoning"] = [r for _, r in results_2]

n_unparsed = test_methods_2["methods_pred"].isna().sum()
print(f"\nUnparsed: {n_unparsed} / {len(test_methods_2)}")

valid_2 = test_methods_2.dropna(subset=["methods_pred"]).copy()

M_2 = np.zeros((len(METHODS_LABELS), len(METHODS_LABELS)), dtype=int)
for gt_labels, pred_labels in zip(valid_2["methods_gt"], valid_2["methods_pred"]):
    for g in gt_labels:
        for p in pred_labels:
            if g in METHODS_LABELS and p in METHODS_LABELS:
                M_2[METHODS_LABELS.index(g), METHODS_LABELS.index(p)] += 1

print("\nRows = annotator, Cols = Gemma (section-picking + thinking)\n")
print(pd.DataFrame(M_2, index=METHODS_LABELS, columns=METHODS_LABELS).to_string())

exact_2 = sum(set(g) == set(p) for g, p in zip(valid_2["methods_gt"], valid_2["methods_pred"]))
print(f"\nExact-match (section-picking + thinking): {exact_2}/{len(valid_2)} = {exact_2/len(valid_2):.1%}")

Full text:    65 / 84
Has headers:  65 / 84

Phase B done. Distribution of #headers picked:
picked
0    25
1    20
2    12
3    14
4    13

Unparsed: 0 / 84

Rows = annotator, Cols = Gemma (section-picking + thinking)

                 qual  quant  both  descriptive  spatial/mapping  unclear
qual               23      0     0            1                0        0
quant               1     14     2            2                6        0
both                1      1     4            1                1        0
descriptive         4      0     0            6                0        1
spatial/mapping     0      1     1            0                4        0
unclear             3      1     0            3                0        3

Exact-match (section-picking + thinking): 54/84 = 64.3%


In [14]:
# Compare to the abstract-only thinking run
joined = valid_t.merge(valid_2, on="doi", suffixes=("_abstract", "_sections"))
flipped = joined[joined["methods_pred_abstract"].astype(str) != joined["methods_pred_sections"].astype(str)]
print(f"Predictions changed vs abstract-only: {len(flipped)} / {len(joined)}")

Predictions changed vs abstract-only: 2 / 83


In [15]:
# Inspect every flipped paper: abstract-only pred + reasoning vs section-picking pred + reasoning,
# plus the actual section content gemma read in phase C.

abs_lookup = valid_t.set_index("doi")[["methods_pred", "methods_reasoning"]].to_dict(orient="index")

flipped_rows = []
for _, r in valid_2.iterrows():
    abs_data = abs_lookup.get(r["doi"])
    if abs_data is None:
        continue
    if str(abs_data["methods_pred"]) != str(r["methods_pred"]):
        flipped_rows.append((r, abs_data))

print(f"Flipped predictions: {len(flipped_rows)}\n")

for r, abs_data in flipped_rows:
    gt = r["methods_gt"][0] if r["methods_gt"] else "?"
    abs_pred = abs_data["methods_pred"][0] if abs_data["methods_pred"] else "?"
    sec_pred = r["methods_pred"][0] if r["methods_pred"] else "?"
    direction = "OK->BAD" if abs_pred == gt and sec_pred != gt else ("BAD->OK" if abs_pred != gt and sec_pred == gt else "BAD->BAD")
    print("=" * 80)
    print(f"{direction}  GT={gt}   abstract->{abs_pred}   sections->{sec_pred}")
    print(f"TITLE:  {r['title']}")
    print(f"DOI:    {r['doi']}")
    print(f"PICKED: {r['picked']}")
    print(f"\n--- ABSTRACT-ONLY THOUGHT ---\n{(abs_data['methods_reasoning'] or '')[:700]}")
    print(f"\n--- SECTION-PICKING THOUGHT ---\n{(r['methods_reasoning'] or '')[:700]}")
    print(f"\n--- SECTION TEXT GEMMA SAW (truncated per-section to 400 chars) ---")
    for h in r["picked"]:
        body = r["sections"].get(h) or next((v for k, v in r["sections"].items() if k.lower() == h.lower()), None)
        if body:
            print(f"  ## {h}")
            print(f"    {body[:400]}{'...' if len(body) > 400 else ''}")
    print()

# Summary of flip direction
n_helped = sum(1 for r, a in flipped_rows
               if a["methods_pred"] and r["methods_pred"]
               and a["methods_pred"][0] != r["methods_gt"][0]
               and r["methods_pred"][0] == r["methods_gt"][0])
n_hurt = sum(1 for r, a in flipped_rows
             if a["methods_pred"] and r["methods_pred"]
             and a["methods_pred"][0] == r["methods_gt"][0]
             and r["methods_pred"][0] != r["methods_gt"][0])
n_lateral = len(flipped_rows) - n_helped - n_hurt
print(f"Flip direction summary:  helped={n_helped}  hurt={n_hurt}  lateral (wrong->wrong)={n_lateral}")


Flipped predictions: 2

BAD->OK  GT=qual   abstract->both   sections->qual
TITLE:  Unintended Developments: Gender, Environment, and Collective Governance in a Mexican Ejido
DOI:    10.1080/00045608.2014.910075
PICKED: ['Study Site and Methods']

--- ABSTRACT-ONLY THOUGHT ---
*   Title: "Unintended Developments: Gender, Environment, and Collective Governance in a Mexican Ejido"
    *   Abstract: Mentions "Using ethnographic and survey data collected in a Veracruz ejido". Discusses "novel subjectivities and practices", "men acted as self-imagined private property owners", "women became registered land managers".
    *   Study Site and Methods section:
        *   "employed an ethnographic approach"
        *   "conducted twenty-four semistructured interviews"
        *   "Twenty-one additional interviews were carried out with state actors"
        *   "Participant observation was another important method"
        *   "draws from household survey data... conducted 

--- SECTION-PICKING T

In [19]:
# Export every paper with both abstract-only and section-picking runs to
# frontend/src/lib/data/flipped.json. Each record carries flags so the UI can
# filter (flipped/agree, correct/wrong, has GT, etc.).

import json
from pathlib import Path

OUT = Path("/gpfs1/home/j/s/jstonge1/rural-geog-classif/frontend/src/lib/data/flipped.json")

records = []
for _, r in valid_2.iterrows():
    abs_data = abs_lookup.get(r["doi"]) or {}
    gt = r["methods_gt"][0] if r["methods_gt"] else None
    abs_pred = abs_data.get("methods_pred", [None])[0] if abs_data.get("methods_pred") else None
    sec_pred = r["methods_pred"][0] if r["methods_pred"] else None

    has_gt = gt is not None
    flipped = (abs_pred is not None and sec_pred is not None and abs_pred != sec_pred)
    correct_abstract = (abs_pred == gt) if has_gt and abs_pred is not None else None
    correct_sections = (sec_pred == gt) if has_gt and sec_pred is not None else None

    if not has_gt:
        direction = "no-gt"
    elif not flipped:
        direction = "agree-correct" if correct_sections else "agree-wrong"
    else:
        if correct_abstract and not correct_sections:
            direction = "flip-ok-bad"
        elif not correct_abstract and correct_sections:
            direction = "flip-bad-ok"
        else:
            direction = "flip-lateral"

    picked_sections = {h: (r["sections"].get(h) or
                           next((v for k, v in r["sections"].items() if k.lower() == h.lower()), ""))
                       for h in r["picked"]}

    records.append({
        "doi": r["doi"],
        "title": r["title"],
        "abstract": r.get("abstract", "") or "",
        "direction": direction,
        "flipped": flipped,
        "has_gt": has_gt,
        "correct_abstract": correct_abstract,
        "correct_sections": correct_sections,
        "annotator": list(r["methods_gt"]) if r["methods_gt"] else [],
        "pred_abstract": abs_pred,
        "pred_sections": sec_pred,
        "picked": list(r["picked"]),
        "sections": picked_sections,
        "reasoning_abstract": abs_data.get("methods_reasoning", "") or "",
        "reasoning_sections": r.get("methods_reasoning", "") or "",
    })

# Summary
from collections import Counter
print(f"Exporting {len(records)} records")
print("Direction breakdown:", dict(Counter(r["direction"] for r in records)))

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(records, indent=2, ensure_ascii=False))
print(f"Wrote {OUT}")


Exporting 84 records
Direction breakdown: {'agree-wrong': 29, 'agree-correct': 53, 'flip-bad-ok': 1, 'flip-lateral': 1}
Wrote /gpfs1/home/j/s/jstonge1/rural-geog-classif/frontend/src/lib/data/flipped.json
